# AllMusic Scraper

## Kako je notebook organizovan

1. Instalacija.
2. Podesavanja (lista izvodjaca, putanje, kolone za CSV).
3. Funkcije za citanje/parsiranje HTML-a.
4. Funkcije za otvaranje stranica uz zaobilazenje Cloudflare-a (SeleniumBase
   UC Mode).
5. Glavna funkcija za scraping + pravljenje CSV-a.
6. **TEST na "The Verve"** (mali izvodjac - 3 albuma, 36 pesama).
7. Pun scraping svih izvodjaca.
8. Pregled CSV-a.


In [ ]:
# 1) Instalacija
# SeleniumBase sam preuzima i upravlja odgovarajucim chromedriver-om
!pip install seleniumbase beautifulsoup4 pandas pyautogui --break-system-packages -q


In [8]:
# 2) Podesavanja
import json
import random
import re
import time
import unicodedata
from pathlib import Path
from urllib.parse import urljoin

from bs4 import BeautifulSoup

ARTIST_URLS = {
    "Ash": "https://www.allmusic.com/artist/ash-mn0000615304",
    "Blur": "https://www.allmusic.com/artist/blur-mn0000758444",
    "Brett Anderson": "https://www.allmusic.com/artist/brett-anderson-mn0000621415",
    "Damon Albarn": "https://www.allmusic.com/artist/damon-albarn-mn0000668164",
    "Echobelly": "https://www.allmusic.com/artist/echobelly-mn0000171367",
    "Elastica": "https://www.allmusic.com/artist/elastica-mn0000797431",
    "Jarvis Cocker": "https://www.allmusic.com/artist/jarvis-cocker-mn0000808871",
    "Liam Gallagher": "https://www.allmusic.com/artist/liam-gallagher-mn0000225369",
    "Lush": "https://www.allmusic.com/artist/lush-mn0000169099",
    "Manic Street Preachers": "https://www.allmusic.com/artist/manic-street-preachers-mn0000954964",
    "Menswear": "https://www.allmusic.com/artist/menswear-mn0000422012",
    "Oasis": "https://www.allmusic.com/artist/oasis-mn0000393345",
    "Paul Weller": "https://www.allmusic.com/artist/paul-weller-mn0000029791",
    "Paul Weller, Damon Albarn": "https://www.allmusic.com/artist/paul-weller-mn0000029791",
    "Pulp": "https://www.allmusic.com/artist/pulp-mn0000308645",
    "Shed Seven": "https://www.allmusic.com/artist/shed-seven-mn0000009490",
    "Sleeper": "https://www.allmusic.com/artist/sleeper-mn0000022703",
    "Suede": "https://www.allmusic.com/artist/suede-mn0000586694",
    "Supergrass": "https://www.allmusic.com/artist/supergrass-mn0000031556",
    "The Charlatans": "https://www.allmusic.com/artist/the-charlatans-mn0000068283",
    "The La's": "https://www.allmusic.com/artist/the-las-mn0000104455",
    "The Verve": "https://www.allmusic.com/artist/the-verve-mn0000575522",
    "Travis": "https://www.allmusic.com/artist/travis-mn0000013508",
}

BASE_DIR = Path.cwd()
PROFILE_DIR = BASE_DIR / "uc_profile_allmusic"
CACHE_DIR = BASE_DIR / "allmusic_cache_v2"
CACHE_DIR.mkdir(exist_ok=True)

ALBUMS_CACHE = CACHE_DIR / "albums.json"
SONGS_CACHE = CACHE_DIR / "songs.json"
ARTISTS_CACHE = CACHE_DIR / "artists.json"
FAILED_CACHE = CACHE_DIR / "failed_urls.json"    # URL-ovi koji nisu uspeli - za kasniji retry
LOG_FILE = CACHE_DIR / "scrape_log.txt"
OUTPUT_CSV = BASE_DIR / "allmusic.csv"

DELAY_MIN = 3.0
DELAY_MAX = 7.0

# UC Mode zahteva da putanja profila NE sadrzi razmake (Windows ogranicenje).
USE_PERSISTENT_PROFILE = " " not in str(PROFILE_DIR)
if not USE_PERSISTENT_PROFILE:
    print("UPOZORENJE: putanja radnog foldera sadrzi razmak:")
    print(f"  {BASE_DIR}")
    print("UC Mode profil se NECE trajno cuvati izmedju pokretanja.")
    print("Preporuka: premesti notebook u folder bez razmaka, npr. C:\\allmusic\\")

CSV_COLUMNS = [
    "artist_name", "album_title", "song_title", "track_number", "spotify_id",
    "allmusic_artist_url", "allmusic_song_url", "allmusic_song_genre",
    "allmusic_song_style", "allmusic_song_mood", "allmusic_song_theme",
    "allmusic_composers", "allmusic_track_pick", "allmusic_album_url",
    "allmusic_album_genre", "allmusic_album_style", "allmusic_album_mood",
    "allmusic_album_theme",
    "allmusic_album_duration", "allmusic_recording_location", "allmusic_release_date",
    "allmusic_match_status", "allmusic_match_score", "allmusic_match_type",
]
print("Config OK. Radni folder:", BASE_DIR)


Config OK. Radni folder: C:\Users\Milica\Desktop\FON\diplomski


In [9]:
# 3) Pomocne funkcije za citanje/parsiranje HTML-a (bez browsera - cist tekst)

def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def load_json(path):
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            log(f"UPOZORENJE: {path} je osteceen/nevalidan JSON, krece se od praznog.")
            return {}
    return {}

def save_json(path, data, retries=6, delay=0.5):
    # Na Windows-u os.replace() (preimenovanje .tmp -> .json) ume povremeno da
    # baci "PermissionError: Access is denied" ako neki DRUGI proces bas u tom
    # trenutku drzi fajl otvoren - najcesce OneDrive (ako je folder unutar
    # Desktop-a koji OneDrive sinhronizuje) ili antivirus koji skenira novi
    # fajl. Skoro uvek je to KRATKOTRAJNO, pa probamo ponovo par puta umesto
    # da srusimo ceo scraping zbog jednog neuspelog cuvanja.
    tmp = path.with_suffix(".tmp")
    payload = json.dumps(data, ensure_ascii=False, indent=2)
    for attempt in range(1, retries + 1):
        try:
            tmp.write_text(payload, encoding="utf-8")
            tmp.replace(path)
            return
        except PermissionError as e:
            if attempt == retries:
                log(f"UPOZORENJE: ne mogu da snimim {path.name} posle {retries} pokusaja ({e}). "
                    "Nastavljam scraping dalje - ovi podaci ce se ipak snimiti cim sledece "
                    "cuvanje uspe (sve je i dalje u memoriji).")
                return
            time.sleep(delay * attempt)

def normalize_text(s):
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", s)
    return re.sub(r"\s+", " ", s).strip()

def text_lines(soup):
    raw = soup.get_text("\n")
    lines = [normalize_text(l) for l in raw.split("\n")]
    return [l for l in lines if l]

def value_after_label(lines, label, max_lookahead=6, stop_at_labels=None):
    stop_at_labels = stop_at_labels or []
    for i, line in enumerate(lines):
        if line.strip().lower() == label.strip().lower():
            for j in range(i + 1, min(i + 1 + max_lookahead, len(lines))):
                candidate = lines[j]
                if candidate.strip().lower() in [s.lower() for s in stop_at_labels]:
                    return ""
                return candidate
    return ""

def values_block_after_label(lines, label, stop_labels, use_last=False):
    # use_last=True: koristi POSLEDNJE pojavljivanje labela u tekstu, ne prvo.
    # Potrebno je za "Moods"/"Themes" na stranici albuma, jer se ta ista dva
    # naziva ("Moods", "Themes") pojavljuju i u gornjem meniju sajta (na
    # SVAKOJ stranici, kao opsti spisak svih raspolozivih tagova) PRE nego
    # sto se pojavi stvarna "Moods and Themes" sekcija konkretnog albuma pri
    # dnu stranice. Prvo pojavljivanje bi zato uvek pogresno "pogodilo" meni,
    # ne stvarne podatke
    stop_lower = [s.lower() for s in stop_labels]
    label_lower = label.strip().lower()
    if use_last:
        idx_matches = [i for i, l in enumerate(lines) if l.strip().lower() == label_lower]
        if not idx_matches:
            return ""
        start = idx_matches[-1] + 1
        collected = []
        for line in lines[start:]:
            low = line.strip().lower()
            if low in stop_lower:
                break
            collected.append(line)
        joined = normalize_text(" ".join(collected))
        joined = re.sub(r"\s+,", ",", joined)
        return joined

    collecting = False
    collected = []
    for line in lines:
        low = line.strip().lower()
        if not collecting:
            if low == label_lower:
                collecting = True
            continue
        if low in stop_lower:
            break
        collected.append(line)
    joined = normalize_text(" ".join(collected))
    joined = re.sub(r"\s+,", ",", joined)
    return joined

def extract_rating_best_effort(soup, label_text):
    label_node = soup.find(string=re.compile(rf"^\s*{re.escape(label_text)}\s*$", re.I))
    if not label_node:
        return ""
    container = label_node.parent
    search_scope = []
    node = container
    for _ in range(4):
        if node is None:
            break
        search_scope.append(node)
        node = node.parent

    for scope in search_scope:
        if scope is None:
            continue
        for el in scope.find_all(True):
            for attr in ("data-rating", "data-value", "aria-valuenow", "data-score"):
                if el.has_attr(attr):
                    val = el[attr]
                    try:
                        return str(float(val))
                    except (TypeError, ValueError):
                        pass
            if el.has_attr("aria-label"):
                m = re.search(r"(\d+(\.\d+)?)\s*(out of|/)\s*5", el["aria-label"], re.I)
                if m:
                    return m.group(1)
            cls = " ".join(el.get("class", []))
            m = re.search(r"(?:rating|stars?)[-_](\d+)(?:[-_](\d+))?", cls, re.I)
            if m:
                whole = m.group(1)
                frac = m.group(2)
                if frac:
                    return f"{whole}.{frac}"
                try:
                    n = int(whole)
                    return str(n / 2) if n > 5 else str(n)
                except ValueError:
                    pass
            style = el.get("style", "")
            m = re.search(r"width:\s*(\d+(?:\.\d+)?)%", style)
            if m:
                return str(round(float(m.group(1)) / 20, 2))
    return ""

def section_elements(soup, start_id_regex, end_id_regexes):
    # Vrati listu SVIH tagova IZMEDJU elementa ciji id odgovara start_id_regex
    # i sledeceg elementa ciji id odgovara nekoj od end_id_regexes (ne
    # ukljucujuci kraj). Ovo koristimo da ogranicimo pretragu SAMO na
    # diskografiju, a ne na celu stranicu.
    start = soup.find(attrs={"id": re.compile(start_id_regex, re.I)})
    if not start:
        return None
    collected = []
    for el in start.find_all_next():
        el_id = el.get("id", "") if hasattr(el, "get") else ""
        if el_id and any(re.search(p, el_id, re.I) for p in end_id_regexes):
            break
        collected.append(el)
    return collected

def parse_discography(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    albums, seen = [], set()

    section = section_elements(soup, r"discography", [r"songs", r"credits"])
    if section is None:
        log("UPOZORENJE: nisam nasao #discography sekciju, pretrazujem celu stranicu (moze pokupiti visak linkova).")
        search_space = soup.find_all(True)
    else:
        search_space = section

    for a in search_space:
        if getattr(a, "name", None) != "a" or not a.get("href"):
            continue
        href = a["href"]
        if "/album/" not in href or "/album/release/" in href:
            continue
        full_url = urljoin(base_url, href).split("#")[0]
        if full_url in seen:
            continue
        title = normalize_text(a.get_text())
        if not title:
            continue
        seen.add(full_url)
        albums.append({"url": full_url, "title": title})

    return albums

def parse_tracklist(soup, album_url, artist_name, artist_url):
    tracks = []
    heading = soup.find(id=re.compile(r"trackListing", re.I))
    container = heading.find_parent() if heading else soup
    if container is None:
        container = soup

    raw_links = [a for a in container.find_all("a", href=True)
                 if re.search(r"/song/[^/]+-mt\d+", a["href"])]

    # AllMusic ponekad ima DVA linka za istu pesmu u tracklisti: jedan sa
    # naslovom, i jedan "Song Review" ikonica-link sa ISTIM href-om ali
    # PRAZNIM tekstom (pojavljuje se samo kod pesama koje imaju recenziju).
    # Bez dedupovanja ovo pravi duple redove sa praznim song_title.
    by_href = {}
    order = []
    for a in raw_links:
        href = a["href"]
        if href not in by_href:
            by_href[href] = a
            order.append(href)
        elif not normalize_text(by_href[href].get_text()):
            by_href[href] = a
    song_links = [by_href[href] for href in order]

    for idx, a in enumerate(song_links):
        title = normalize_text(a.get_text())
        href = urljoin(album_url, a["href"]).split("#")[0]

        artist_links = []
        stop_href = song_links[idx + 1]["href"] if idx + 1 < len(song_links) else None
        for sib in a.find_all_next():
            if sib.name == "a" and sib.get("href") == stop_href:
                break
            if sib.name == "a" and "/artist/" in sib.get("href", ""):
                artist_links.append(sib)
            if sib.name == "a" and sib is not a and "/song/" in sib.get("href", "") and sib.get("href") != a.get("href"):
                break

        composer_names = []
        for al in artist_links:
            name = normalize_text(al.get_text())
            if not name:
                continue
            if name.lower() == artist_name.lower() or al.get("href", "").rstrip("/") == artist_url.rstrip("/"):
                continue
            composer_names.append(name)

        after_text_parts = []
        count = 0
        for sib in a.find_all_next(string=True):
            after_text_parts.append(sib)
            count += 1
            if count > 15:
                break
        joined = " ".join(normalize_text(p) for p in after_text_parts)
        m = re.search(r"\b(\d{1,2}:\d{2})\b", joined)
        duration = m.group(1) if m else ""

        track_pick = "unknown"
        anc = a
        found_pick = False
        found_marker = False
        for _ in range(5):
            if anc is None:
                break
            cls = " ".join(anc.get("class", [])) if hasattr(anc, "get") else ""
            if re.search(r"pick|highlight|featured", cls, re.I):
                found_pick = True
                found_marker = True
                break
            style = anc.get("style", "") if hasattr(anc, "get") else ""
            if "color" in style.lower():
                found_marker = True
            anc = anc.parent
        if found_marker:
            track_pick = "yes" if found_pick else "no"

        tracks.append({
            "track_number": idx + 1, "song_title": title, "song_url": href,
            "composers_on_album": ", ".join(dict.fromkeys(composer_names)),
            "duration": duration, "track_pick": track_pick,
        })
    return tracks

def parse_album_page(html, album_url, artist_name, artist_url):
    soup = BeautifulSoup(html, "html.parser")
    lines = text_lines(soup)
    known_labels = ["Release Date", "Duration", "Genre", "Styles", "Recording Date",
                    "Recording Location", "AllMusic Rating", "User Rating", "Your Rating"]

    release_date = value_after_label(lines, "Release Date")
    duration = value_after_label(lines, "Duration")
    genre = values_block_after_label(lines, "Genre", known_labels)
    styles = values_block_after_label(lines, "Styles", known_labels)
    recording_location = value_after_label(lines, "Recording Location") or value_after_label(lines, "Recording Date")

    album_mood = values_block_after_label(lines, "Moods", ["Themes", "Submit Corrections"], use_last=True)
    album_theme = values_block_after_label(lines, "Themes", ["Submit Corrections"], use_last=True)

    tracks = parse_tracklist(soup, album_url, artist_name, artist_url)
    h1 = soup.find("h1")
    return {
        "album_url": album_url,
        "album_title": normalize_text(h1.get_text() if h1 else "").split(" [")[0],
        "release_date": release_date, "duration": duration, "genre": genre,
        "style": styles, "mood": album_mood, "theme": album_theme,
        "recording_location": recording_location, "tracks": tracks,
    }

def parse_song_page(html, song_url):
    soup = BeautifulSoup(html, "html.parser")
    lines = text_lines(soup)
    known_labels = ["Composed by", "Release Year", "Genre", "Styles", "Song Review",
                    "Song Genre & Styles", "Song Moods & Themes"]
    composed_by = values_block_after_label(lines, "Composed by", known_labels)
    genre = values_block_after_label(lines, "Genre", known_labels)
    styles = values_block_after_label(lines, "Styles", ["Appears On", "Song Review"] + known_labels)

    # VAZNO: NE pretrazujemo CELU stranicu za /mood//theme/ linkove, jer se u
    # gornjem meniju sajta (isti na SVAKOJ stranici) nalazi fiksan spisak SVIH
    # raspolozivih moods/themes tagova - kad bismo pretrazili celu stranicu,
    # svaka pesma bi dobila taj isti opsti spisak umesto svojih stvarnih
    # tagova. Menu se uvek nalazi PRE oznake "Styles" u toku stranice, pa
    # trazimo linkove SAMO POSLE nje (find_all_next).
    styles_marker = soup.find(string=re.compile(r"^\s*Styles\s*$", re.I))
    mood_theme_links = styles_marker.find_all_next("a", href=True) if styles_marker else soup.find_all("a", href=True)

    moods, themes = [], []
    for a in mood_theme_links:
        href = a["href"]
        if "/mood/" in href:
            name = re.sub(r"\s*\(\d+\)\s*$", "", normalize_text(a.get_text()))
            if name:
                moods.append(name)
        elif "/theme/" in href:
            name = re.sub(r"\s*\(\d+\)\s*$", "", normalize_text(a.get_text()))
            if name:
                themes.append(name)

    return {
        "song_url": song_url, "composed_by": composed_by, "genre": genre, "style": styles,
        "mood": ", ".join(dict.fromkeys(moods)), "theme": ", ".join(dict.fromkeys(themes)),
    }

print("Parsing funkcije ucitane (sa ispravkom diskografije).")


Parsing funkcije ucitane (sa ispravkom diskografije).


In [10]:
# 4) Funkcije za otvaranje stranica uz zaobilazenje Cloudflare-a (SeleniumBase UC Mode)
from seleniumbase import SB

CLOUDFLARE_MARKERS = ["verifying you are human", "just a moment", "checking your browser"]

# Brojac uzastopnih PUNIH neuspeha (kad ni automatski klik ni rucno resavanje
# ne upale) - koristimo ga da napravimo duzu pauzu ako Cloudflare pocne cesto
# da blokira zaredom (znak da sesija privremeno "izgubila poverenje").
_consecutive_blocks = [0]

def is_cloudflare_challenge(text):
    text = (text or "").lower()
    return any(m in text for m in CLOUDFLARE_MARKERS)

def fetch(sb, url, max_retries=2, interactive=False):
    # Ucitaj URL preko UC Mode-a (uc_open_with_reconnect), proveri da li se
    # pojavio Cloudflare izazov, i ako jeste - prvo probaj automatski klik
    # (uc_gui_click_captcha). Ako ni to ne uspe:
    #   - interactive=True  -> zatrazi RUCNU intervenciju (obican Python
    #     input(), jer se SeleniumBase-ov ugradjeni "breakpoint" mehanizam
    #     pokazao nepouzdanim u Jupyter-u - nije stvarno cekao unos).
    #   - interactive=False -> ODUSTANI od ovog URL-a i vrati None (pozivalac
    #     ce ga zabeleziti za kasniji retry_failed() poziv). Ovo omogucava da
    #     pustis scraping da radi BEZ da sedis pored racunara cele vreme.
    for attempt in range(1, max_retries + 1):
        try:
            sb.uc_open_with_reconnect(url, reconnect_time=4)
            sb.sleep(1)
            html = sb.get_page_source()

            if is_cloudflare_challenge(html) or is_cloudflare_challenge(sb.get_title()):
                log(f"Cloudflare izazov na {url}, pokusavam automatski klik...")
                try:
                    sb.uc_gui_click_captcha()
                except Exception as e:
                    log(f"uc_gui_click_captcha nije uspeo/nije bio potreban: {e}")
                sb.sleep(3)
                html = sb.get_page_source()

            if is_cloudflare_challenge(html) and interactive:
                print("\n" + "=" * 70)
                print(f"Cloudflare se ne resava automatski na: {url}")
                print("Resi proveru RUCNO u browser prozoru koji je otvoren (sacekaj")
                print("da se ucita normalna AllMusic stranica), pa se vrati ovde.")
                input("Kada stranica bude normalna, pritisni ENTER da nastavim... ")
                sb.sleep(1)
                html = sb.get_page_source()

            if not is_cloudflare_challenge(html):
                _consecutive_blocks[0] = 0
                return html

            log(f"Cloudflare i dalje blokira {url} (pokusaj {attempt}/{max_retries}).")
        except Exception as e:
            log(f"Greska pri ucitavanju {url} (pokusaj {attempt}/{max_retries}): {e}")
            sb.sleep(3 * attempt)

    _consecutive_blocks[0] += 1
    log(f"NEUSPEH: odustajem od {url} (uzastopnih neuspeha: {_consecutive_blocks[0]}).")
    if _consecutive_blocks[0] >= 3:
        cooldown = 90
        log(f"3+ uzastopna neuspeha - pravim pauzu od {cooldown}s da se sesija 'ohladi'.")
        time.sleep(cooldown)
        _consecutive_blocks[0] = 0
    return None

def human_delay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

print("Browser/Cloudflare funkcije ucitane (SeleniumBase UC Mode).")


Browser/Cloudflare funkcije ucitane (SeleniumBase UC Mode).


In [11]:
# 5) Glavna scrape() funkcija + build_csv()
# Napomena: SB(...) OTVARA browser samo JEDNOM na pocetku i koristi ga za sve
# stranice u ovom pozivu - to je brze i manje upadljivo nego otvaranje novog
# browsera za svaku stranicu.

def load_failed():
    data = load_json(FAILED_CACHE)
    return data if isinstance(data, list) else []

def record_failed(item):
    # item je recnik npr. {"type": "song", "url": ...} - cuvamo da bismo
    # kasnije mogli da probamo TACNO te URL-ove ponovo preko retry_failed().
    failed = load_failed()
    if not any(f.get("url") == item.get("url") for f in failed):
        failed.append(item)
        save_json(FAILED_CACHE, failed)

def remove_failed(url):
    failed = load_failed()
    failed = [f for f in failed if f.get("url") != url]
    save_json(FAILED_CACHE, failed)


def scrape(artist_filter=None, headless=False, interactive=False):
    # interactive=False (podrazumevano) znaci: ako Cloudflare blokira neku
    # stranicu i automatski klik ne pomogne, skripta NECE cekati da ti resis
    # rucno - zabelezice taj URL i nastaviti dalje.
    if headless:
        log("UPOZORENJE: UC Mode se lako otkriva u headless modu - preporucuje se headless=False.")

    artists_cache = load_json(ARTISTS_CACHE)
    albums_cache = load_json(ALBUMS_CACHE)
    songs_cache = load_json(SONGS_CACHE)

    targets = {k: v for k, v in ARTIST_URLS.items() if not artist_filter or k in artist_filter}
    if not targets:
        log("Nijedan izvodjac se ne poklapa sa filterom, prekidam.")
        return

    sb_kwargs = dict(uc=True, test=True, incognito=False, headless=headless)
    if USE_PERSISTENT_PROFILE:
        sb_kwargs["user_data_dir"] = str(PROFILE_DIR)

    with SB(**sb_kwargs) as sb:
        for artist_name, artist_url in targets.items():
            log(f"=== Izvodjac: {artist_name} ===")

            if artist_name in artists_cache:
                album_list = artists_cache[artist_name]["albums"]
                log(f"  (iz kesa: {len(album_list)} albuma)")
            else:
                html = fetch(sb, artist_url, interactive=interactive)
                if html is None:
                    log(f"  Preskacem {artist_name} - stranica se nije ucitala.")
                    record_failed({"type": "artist", "url": artist_url, "artist_name": artist_name})
                    continue
                album_list = parse_discography(html, artist_url)
                artists_cache[artist_name] = {"url": artist_url, "albums": album_list}
                save_json(ARTISTS_CACHE, artists_cache)
                log(f"  Pronadjeno {len(album_list)} albuma.")
                human_delay()

            for album in album_list:
                album_url = album["url"]
                if album_url in albums_cache:
                    continue
                log(f"  Album: {album['title']} ({album_url})")
                html = fetch(sb, album_url, interactive=interactive)
                if html is None:
                    record_failed({"type": "album", "url": album_url, "artist_name": artist_name, "artist_url": artist_url})
                    continue
                parsed = parse_album_page(html, album_url, artist_name, artist_url)
                parsed["artist_name"] = artist_name
                parsed["artist_url"] = artist_url
                albums_cache[album_url] = parsed
                save_json(ALBUMS_CACHE, albums_cache)
                human_delay()

            song_urls = set()
            for album in album_list:
                album_data = albums_cache.get(album["url"])
                if not album_data:
                    continue
                for t in album_data.get("tracks", []):
                    song_urls.add(t["song_url"])

            log(f"  {len(song_urls)} jedinstvenih pesama za obradu.")
            for song_url in song_urls:
                if song_url in songs_cache:
                    continue
                html = fetch(sb, song_url, interactive=interactive)
                if html is None:
                    record_failed({"type": "song", "url": song_url})
                    continue
                songs_cache[song_url] = parse_song_page(html, song_url)
                save_json(SONGS_CACHE, songs_cache)
                human_delay()

    failed_count = len(load_failed())
    log(f"Scraping zavrsen (ili prekinut). Neuspesnih URL-ova: {failed_count}.")
    if failed_count:
        log("Pokreni retry_failed() da (uz tvoje prisustvo) pokusas ponovo te URL-ove.")
    log("Pokreni build_csv() da napravis/azuriras allmusic.csv.")


def retry_failed(headless=False, interactive=True):
    # Ponovo pokusaj SAMO one URL-ove koji ranije nisu uspeli. Podrazumevano
    # interactive=True
    failed = load_failed()
    if not failed:
        print("Nema neuspesnih URL-ova - nema sta da se ponovi.")
        return

    print(f"Pokusavam ponovo {len(failed)} URL-ova koji ranije nisu uspeli...")
    albums_cache = load_json(ALBUMS_CACHE)
    songs_cache = load_json(SONGS_CACHE)
    artists_cache = load_json(ARTISTS_CACHE)

    sb_kwargs = dict(uc=True, test=True, incognito=False, headless=headless)
    if USE_PERSISTENT_PROFILE:
        sb_kwargs["user_data_dir"] = str(PROFILE_DIR)

    with SB(**sb_kwargs) as sb:
        for item in list(failed):
            url = item["url"]
            log(f"Retry [{item['type']}]: {url}")
            html = fetch(sb, url, interactive=interactive)
            if html is None:
                log(f"  I dalje neuspesno: {url}")
                human_delay()
                continue

            if item["type"] == "song":
                songs_cache[url] = parse_song_page(html, url)
                save_json(SONGS_CACHE, songs_cache)
            elif item["type"] == "album":
                parsed = parse_album_page(html, url, item["artist_name"], item["artist_url"])
                parsed["artist_name"] = item["artist_name"]
                parsed["artist_url"] = item["artist_url"]
                albums_cache[url] = parsed
                save_json(ALBUMS_CACHE, albums_cache)
            elif item["type"] == "artist":
                album_list = parse_discography(html, url)
                artists_cache[item["artist_name"]] = {"url": url, "albums": album_list}
                save_json(ARTISTS_CACHE, artists_cache)

            remove_failed(url)
            log(f"  Uspelo: {url}")
            human_delay()

    still_failed = len(load_failed())
    print(f"Gotovo. Jos uvek neuspesnih URL-ova: {still_failed}.")


def dump_html(url, headless=False):
    sb_kwargs = dict(uc=True, test=True, incognito=False, headless=headless)
    if USE_PERSISTENT_PROFILE:
        sb_kwargs["user_data_dir"] = str(PROFILE_DIR)
    with SB(**sb_kwargs) as sb:
        html = fetch(sb, url)
    if html is None:
        print("Nije uspelo ucitavanje stranice.")
        return
    out = BASE_DIR / "debug_dump.html"
    out.write_text(html, encoding="utf-8")
    print(f"Sacuvano u {out} - posalji taj fajl za doterivanje selektora.")


def _dedup_tracks(tracks):
    # Odbrambeni dedup NA IZLAZU: ako AllMusic stranica ima dva linka za istu
    # pesmu (naslov + "Song Review" ikonica), pa je u kesu ipak ostalo dva
    # zapisa za isti song_url (jedan sa naslovom, jedan prazan), ovde ih
    # spajamo u JEDAN red - tako da build_csv() UVEK proizvede cist fajl bez
    # praznih song_title redova, bez obzira na to sta je tacno u kesu.
    by_url = {}
    order = []
    for t in tracks:
        url = t.get("song_url")
        if url not in by_url:
            by_url[url] = t
            order.append(url)
        elif not (by_url[url].get("song_title") or "").strip() and (t.get("song_title") or "").strip():
            by_url[url] = t
    return [by_url[u] for u in order]

def build_csv():
    albums_cache = load_json(ALBUMS_CACHE)
    songs_cache = load_json(SONGS_CACHE)
    rows = []
    for album_url, album in albums_cache.items():
        for t in _dedup_tracks(album.get("tracks", [])):
            song = songs_cache.get(t["song_url"], {})
            composers = t.get("composers_on_album") or song.get("composed_by") or ""
            rows.append({
                "artist_name": album.get("artist_name", ""),
                "album_title": album.get("album_title", ""),
                "song_title": t.get("song_title", ""),
                "track_number": t.get("track_number", ""),
                "spotify_id": "",
                "allmusic_artist_url": album.get("artist_url", ""),
                "allmusic_song_url": t.get("song_url", ""),
                "allmusic_song_genre": song.get("genre", ""),
                "allmusic_song_style": song.get("style", ""),
                "allmusic_song_mood": song.get("mood", ""),
                "allmusic_song_theme": song.get("theme", ""),
                "allmusic_composers": composers,
                "allmusic_track_pick": t.get("track_pick", "unknown"),
                "allmusic_album_url": album_url,
                "allmusic_album_genre": album.get("genre", ""),
                "allmusic_album_style": album.get("style", ""),
                "allmusic_album_mood": album.get("mood", ""),
                "allmusic_album_theme": album.get("theme", ""),
                "allmusic_album_duration": album.get("duration", ""),
                "allmusic_recording_location": album.get("recording_location", ""),
                "allmusic_release_date": album.get("release_date", ""),
                "allmusic_match_status": "pending",
                "allmusic_match_score": "",
                "allmusic_match_type": "",
            })
    import csv
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    log(f"Napisano {len(rows)} redova u {OUTPUT_CSV}")

print("scrape(), dump_html() i build_csv() spremni.")


scrape(), dump_html() i build_csv() spremni.


## 6) Test na malom izvodjacu ("The Verve")

"The Verve" ima samo 3-4 albuma / ~40 pesama, pa je idealan za brz test.

Podrazumevano
(`interactive=False`), ako Cloudflare zablokira neku stranicu i automatski
klik ne pomogne, skripta to ZABELEZI i ide dalje na sledecu stranicu, umesto
da ceka unos.


In [ ]:
# 6) Test na jednom (malom) izvodjacu
# scrape(artist_filter=["The Verve"])


In [12]:
# 7) POKRETANJE ZA SVE IZVODJACE (OKO 9-10 SATI)
scrape()


==================== {  File "<frozen runpy>:198:SB} starts ====================
[2026-08-03 10:44:53] === Izvodjac: Ash ===
[2026-08-03 10:44:53]   (iz kesa: 10 albuma)
[2026-08-03 10:44:53]   118 jedinstvenih pesama za obradu.
[2026-08-03 10:44:53] === Izvodjac: Blur ===
[2026-08-03 10:44:53]   (iz kesa: 12 albuma)
[2026-08-03 10:44:53]   213 jedinstvenih pesama za obradu.
[2026-08-03 10:45:02] Cloudflare izazov na https://www.allmusic.com/song/parklife-mt0012432669, pokusavam automatski klik...
[2026-08-03 10:45:14] Cloudflare i dalje blokira https://www.allmusic.com/song/parklife-mt0012432669 (pokusaj 1/2).
[2026-08-03 10:45:23] Cloudflare izazov na https://www.allmusic.com/song/parklife-mt0012432669, pokusavam automatski klik...
[2026-08-03 10:45:34] Cloudflare i dalje blokira https://www.allmusic.com/song/parklife-mt0012432669 (pokusaj 2/2).
[2026-08-03 10:45:34] NEUSPEH: odustajem od https://www.allmusic.com/song/parklife-mt0012432669 (uzastopnih neuspeha: 1).
[2026-08-03 10:50:

In [13]:
# 7b) Ponovi URL-ove koji nisu uspeli tokom scrape()
retry_failed()


Pokusavam ponovo 11 URL-ova koji ranije nisu uspeli...
==================== {  File "<frozen runpy>:198:SB} starts ====================
[2026-08-03 19:23:04] Retry [song]: https://www.allmusic.com/song/parklife-mt0012432669
[2026-08-03 19:23:14]   Uspelo: https://www.allmusic.com/song/parklife-mt0012432669
[2026-08-03 19:23:20] Retry [song]: https://www.allmusic.com/song/end-of-a-century-mt0045931766
[2026-08-03 19:23:29]   Uspelo: https://www.allmusic.com/song/end-of-a-century-mt0045931766
[2026-08-03 19:23:35] Retry [song]: https://www.allmusic.com/song/tree-of-beauty-mt0043957280
[2026-08-03 19:23:44]   Uspelo: https://www.allmusic.com/song/tree-of-beauty-mt0043957280
[2026-08-03 19:23:51] Retry [song]: https://www.allmusic.com/song/go-away-mt0005783455
[2026-08-03 19:23:59]   Uspelo: https://www.allmusic.com/song/go-away-mt0005783455
[2026-08-03 19:24:02] Retry [song]: https://www.allmusic.com/song/lustra-mt0001969955
[2026-08-03 19:24:11]   Uspelo: https://www.allmusic.com/song/lu

In [15]:
# 8) Generisi allmusic.csv iz prikupljenog kesa
build_csv()

import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
print(df.shape)
df.head(20)


[2026-08-03 19:25:50] Napisano 2313 redova u C:\Users\Milica\Desktop\FON\diplomski\allmusic.csv
(2313, 24)


,artist_name,album_title,song_title,track_number,spotify_id,allmusic_artist_url,allmusic_song_url,allmusic_song_genre,allmusic_song_style,allmusic_song_mood,...,allmusic_album_genre,allmusic_album_style,allmusic_album_mood,allmusic_album_theme,allmusic_album_duration,allmusic_recording_location,allmusic_release_date,allmusic_match_status,allmusic_match_score,allmusic_match_type
0,The Verve,A Storm in Heaven The Verve,Star Sail,1,NaN,https://www.allmusic.com/artist/the-verve-mn00...,https://www.allmusic.com/song/star-sail-mt0053...,Pop/Rock,Alternative Pop/Rock Alternative/Indie Rock In...,"Druggy, Reflective, Sensual, Trippy, Restraine...",...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, ...",Dreamy Spacey Cerebral Detached Restrained Tri...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN
1,The Verve,A Storm in Heaven The Verve,Slide Away,2,NaN,https://www.allmusic.com/artist/the-verve-mn00...,https://www.allmusic.com/song/slide-away-mt005...,Pop/Rock,Alternative/Indie Rock Indie Rock Alternative ...,"Brash, Druggy, Reflective, Restrained, Stylish...",...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, ...",Dreamy Spacey Cerebral Detached Restrained Tri...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN
2,The Verve,A Storm in Heaven The Verve,Already There,3,NaN,https://www.allmusic.com/artist/the-verve-mn00...,https://www.allmusic.com/song/already-there-mt...,Pop/Rock,Alternative Pop/Rock Alternative/Indie Rock Sh...,"Druggy, Reflective, Trippy, Restrained, Stylis...",...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, ...",Dreamy Spacey Cerebral Detached Restrained Tri...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN
3,The Verve,A Storm in Heaven The Verve,Beautiful Mind,4,NaN,https://www.allmusic.com/artist/the-verve-mn00...,https://www.allmusic.com/song/beautiful-mind-m...,Pop/Rock,Alternative Pop/Rock Alternative/Indie Rock In...,"Druggy, Reflective, Sensual, Trippy, Restraine...",...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, ...",Dreamy Spacey Cerebral Detached Restrained Tri...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN
4,The Verve,A Storm in Heaven The Verve,The Sun the Sea,5,NaN,https://www.allmusic.com/artist/the-verve-mn00...,https://www.allmusic.com/song/the-sun-the-sea-...,Pop/Rock,Alternative Pop/Rock Alternative/Indie Rock In...,"Brash, Confrontational, Visceral, Volatile",...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, ...",Dreamy Spacey Cerebral Detached Restrained Tri...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN
5,The Verve,A Storm in Heaven The Verve,Virtual World,6,NaN,https://www.allmusic.com/artist/the-verve-mn00...,https://www.allmusic.com/song/virtual-world-mt...,Pop/Rock,Alternative Pop/Rock Alternative/Indie Rock In...,"Druggy, Ethereal, Reflective, Detached, Stylis...",...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, ...",Dreamy Spacey Cerebral Detached Restrained Tri...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN
6,The Verve,A Storm in Heaven The Verve,Make It ‘Til Monday,7,NaN,https://www.allmusic.com/artist/the-verve-mn00...,https://www.allmusic.com/song/make-it-til-mond...,NaN,We currently don’t have any Styles associated ...,NaN,...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, ...",Dreamy Spacey Cerebral Detached Restrained Tri...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN
7,The Verve,A Storm in Heaven The Verve,Blue,8,NaN,https://www.allmusic.com/artist/the-verve-mn00...,https://www.allmusic.com/song/blue-mt0053617240,Pop/Rock,Alternative/Indie Rock Shoegaze Space Rock Ind...,"Brash, Druggy, Trippy, Detached, Sensual, Ethe...",...,Pop/Rock,"Alternative Pop/Rock, Alternative/Indie Rock, ...",Dreamy Spacey Cerebral Detached Restrained Tri...,Night Driving,47:02,NaN,"June 21, 1993",pending,NaN,NaN
8,The Verve,A Storm in Heaven The Verve,Butterfly,9,NaN,https://www.allmusic.com/artist/the-verve-mn00...,https://www.allmusic.com/song/butterfly-mt0053